# D12 · Linux side - SocketCAN, DBC decoding and Industrial I/O

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/02_drivers_protocols/D12_linux_side_socketcan_dbc_decoding_and_industrial.ipynb)

**Track:** Drivers and protocols · **Skill axis (sheet):** Drivers and protocols · **Time:** about 55 min

**Skill Hub concepts:** SocketCAN; Linux Industrial I/O and sysfs

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

**Languages:** each low-level task has an optional C/C++ cell after it. Same tests: write the cell, then uncomment the `lab.check(..., "file.c")` line under it. A check passes if either language passes.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("D12")

## Map: how Linux exposes a CAN bus and an ADC

**SocketCAN:** a CAN controller is a network interface (`can0`); apps open `socket(PF_CAN, SOCK_RAW, CAN_RAW)`
and read or write `struct can_frame`. Many programs can share the bus, and the same tools work for every adapter.

```
 struct can_frame {            offset  size
     canid_t can_id;           0       4    bits 0..28 id, bit 31 EFF (29-bit), bit 30 RTR, bit 29 ERR
     __u8    len;              4       1    (can_dlc)
     __u8    __pad, __res0, len8_dlc;  5    3
     __u8    data[8];          8       8
 };                                    = 16 bytes      (struct canfd_frame: 72 bytes)
```

| Shell (on your own Linux box; Colab cannot load kernel modules) | Effect |
|---|---|
| `sudo ip link set can0 type can bitrate 500000 && sudo ip link set up can0` | configure a real adapter |
| `sudo modprobe vcan && sudo ip link add dev vcan0 type vcan && sudo ip link set up vcan0` | virtual bus for testing |
| `candump -L can0` / `cansend can0 123#E803F4011E00` | log / send |
| `setsockopt(SOL_CAN_RAW, CAN_RAW_FILTER, ...)` | kernel filter: pass if `rx_id & mask == id & mask`; `CAN_INV_FILTER` inverts |

**DBC files** describe signals inside frames: start bit, length, byte order, sign, scale, offset, unit.
`physical = raw * scale + offset`.

**Industrial I/O (IIO):** ADCs, IMUs and DACs appear under `/sys/bus/iio/devices/iio:deviceN/`:
`in_voltage0_raw`, `in_voltage_scale` (shared) or `in_voltage0_scale` (per channel, wins), optional `_offset`.
`processed = (raw + offset) * scale`, in millivolts for voltage and milli-degrees Celsius for temperature.
Buffered capture uses `/dev/iio:deviceN` with triggers instead of one sysfs read per sample.

In [ ]:
from roboprep.checks.d12 import DBC, LOG   # a small DBC and a candump -L log
print(DBC)
print(LOG)

## Task 1: parse a candump log and decode it with the DBC

In [ ]:
import cantools

def parse_candump(text):
    """Return a list of (timestamp, channel, can_id, data_bytes)."""
    raise NotImplementedError  # TODO

def decode_motor(frames, dbc_text):
    """Decode every 0x123 frame; return a list of dicts {Speed, Current, Temperature, Fault}."""
    raise NotImplementedError  # TODO

lab.check("candump", parse_candump, decode_motor)

In [ ]:
%%writefile d12_candump.c
// C/C++ version of Task 1 (optional, same tests). Declarations: roboprep/c/include/d12.h
// Grade it with lab.check("candump", "d12_candump.c"). For C++, name the file .cpp in both places.
#include "d12.h"

size_t parse_candump(const char *text, CandumpFrame *out, size_t out_cap) {
    (void)text; (void)out; (void)out_cap;
    return 0; /* TODO: sscanf each line: (timestamp) channel id#hexdata */
}

size_t decode_motor(const CandumpFrame *frames, size_t n, MotorStatus *out, size_t out_cap) {
    (void)frames; (void)n; (void)out; (void)out_cap;
    return 0; /* TODO: little-endian signals from the DBC: Speed 0|16 signed x0.1, Current 16|16 signed x0.01,
                 Temperature 32|8 unsigned offset -40, Fault bit 40 */
}

In [ ]:
# lab.check("candump", "d12_candump.c")  # uncomment after writing the C/C++ cell above

## Task 2: `struct can_frame` bytes and kernel filter rules

In [ ]:
import struct
CAN_EFF_FLAG, CAN_RTR_FLAG, CAN_ERR_FLAG, CAN_INV_FILTER = 0x80000000, 0x40000000, 0x20000000, 0x20000000

def pack_can_frame(can_id, data, extended=False, rtr=False):
    raise NotImplementedError  # TODO

def filter_pass(rx_can_id, filters):
    """rx_can_id includes flag bits. filters: list of (can_id, can_mask). Pass if any filter matches."""
    raise NotImplementedError  # TODO

lab.check("frames", pack_can_frame, filter_pass)

In [ ]:
%%writefile d12_frames.c
// C/C++ version of Task 2 (optional, same tests). Declarations: roboprep/c/include/d12.h
// Grade it with lab.check("frames", "d12_frames.c"). For C++, name the file .cpp in both places.
#include "d12.h"

void pack_can_frame(uint32_t can_id, const uint8_t *data, size_t len, bool extended, bool rtr, uint8_t *out16) {
    (void)can_id; (void)data; (void)len; (void)extended; (void)rtr;
    memset(out16, 0, 16); /* TODO: can_id with flags (4 bytes), length, 3 padding bytes, 8 data bytes */
}

bool filter_pass(uint32_t rx_can_id, const CanFilter *filters, size_t n) {
    (void)rx_can_id; (void)filters; (void)n;
    return false; /* TODO */
}

In [ ]:
# lab.check("frames", "d12_frames.c")  # uncomment after writing the C/C++ cell above

## Task 3: read an IIO device from sysfs
The checker builds a fake `iio:device0` folder. Return `{"voltage0": mV, ..., "temp": milli_degC}` for every
`in_*_raw` file, using the per-channel scale if present, else the shared one, and the offset if present (else 0).

In [ ]:
import os, re

def read_iio(dev):
    raise NotImplementedError  # TODO

lab.check("iio", read_iio)

In [ ]:
%%writefile d12_iio.c
// C/C++ version of Task 3 (optional, same tests). Declarations: roboprep/c/include/d12.h
// Grade it with lab.check("iio", "d12_iio.c"). For C++, name the file .cpp in both places.
#include "d12.h"

size_t read_iio(const char *dev_dir, IioChannel *out, size_t out_cap) {
    (void)dev_dir; (void)out; (void)out_cap;
    return 0; /* TODO: opendir/readdir; for each in_<kind><idx>_raw: (raw + offset) * scale,
                 channel files first, then the shared in_<kind>_scale / _offset */
}

In [ ]:
# lab.check("iio", "d12_iio.c")  # uncomment after writing the C/C++ cell above

## Checkpoint: Math
**M1 (Remember).** `sizeof(struct can_frame)`?

**M2 (Apply).** `can_id` field value for extended identifier `0x18FF50E5` (with the EFF flag), as an integer?

**M3 (Apply).** `in_voltage0_raw = 2048`, `in_voltage_scale = 0.805664062`. Millivolts?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Evaluate).** Why did Linux model CAN as a network device instead of a character device per adapter?
- a) it is faster on the wire
- b) many programs can share one bus, the kernel filters frames, and the same sockets and tools work for every adapter
- c) CAN frames are IP packets

**K2 (Understand).** An IMU driver exposes both sysfs channels and a buffer. Explain when you would read sysfs and when
the buffer, and what the trigger does.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `d12.h`).
3. In `roboprep/checks/d12.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()